In [4]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder, OrdinalEncoder
from sklearn.pipeline import Pipeline

In [5]:
pd.set_option('display.max_columns', None)

In [6]:
df = pd.read_csv('data/heart_disease.csv')
df.head()

,Age,Gender,Blood Pressure,Cholesterol Level,Exercise Habits,Smoking,Family Heart Disease,Diabetes,BMI,High Blood Pressure,Low HDL Cholesterol,High LDL Cholesterol,Alcohol Consumption,Stress Level,Sleep Hours,Sugar Consumption,Triglyceride Level,Fasting Blood Sugar,CRP Level,Homocysteine Level,Heart Disease Status
0,56.0,Male,153.0,155.0,High,Yes,Yes,No,24.991591,Yes,Yes,No,High,Medium,7.633228,Medium,342.0,NaN,12.969246,12.387250,No
1,69.0,Female,146.0,286.0,High,No,Yes,Yes,25.221799,No,Yes,No,Medium,High,8.744034,Medium,133.0,157.0,9.355389,19.298875,No
2,46.0,Male,126.0,216.0,Low,No,No,No,29.855447,No,Yes,Yes,Low,Low,4.440440,Low,393.0,92.0,12.709873,11.230926,No
3,32.0,Female,122.0,293.0,High,Yes,Yes,No,24.130477,Yes,No,Yes,Low,High,5.249405,High,293.0,94.0,12.509046,5.961958,No
4,60.0,Male,166.0,242.0,Low,Yes,Yes,Yes,20.486289,Yes,No,No,Low,High,7.030971,High,263.0,154.0,10.381259,8.153887,No


In [7]:
yes_no_columns = ['Smoking', 'Family Heart Disease', 'Diabetes', 
                    'High Blood Pressure', 'Low HDL Cholesterol',
                    'High LDL Cholesterol', 'Heart Disease Status']
df[yes_no_columns] = (df[yes_no_columns]=='Yes').astype(int)
df

,Age,Gender,Blood Pressure,Cholesterol Level,Exercise Habits,Smoking,Family Heart Disease,Diabetes,BMI,High Blood Pressure,Low HDL Cholesterol,High LDL Cholesterol,Alcohol Consumption,Stress Level,Sleep Hours,Sugar Consumption,Triglyceride Level,Fasting Blood Sugar,CRP Level,Homocysteine Level,Heart Disease Status
0,56.0,Male,153.0,155.0,High,1,1,0,24.991591,1,1,0,High,Medium,7.633228,Medium,342.0,NaN,12.969246,12.387250,0
1,69.0,Female,146.0,286.0,High,0,1,1,25.221799,0,1,0,Medium,High,8.744034,Medium,133.0,157.0,9.355389,19.298875,0
2,46.0,Male,126.0,216.0,Low,0,0,0,29.855447,0,1,1,Low,Low,4.440440,Low,393.0,92.0,12.709873,11.230926,0
3,32.0,Female,122.0,293.0,High,1,1,0,24.130477,1,0,1,Low,High,5.249405,High,293.0,94.0,12.509046,5.961958,0
4,60.0,Male,166.0,242.0,Low,1,1,1,20.486289,1,0,0,Low,High,7.030971,High,263.0,154.0,10.381259,8.153887,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9995,25.0,Female,136.0,243.0,Medium,1,0,0,18.788791,1,0,1,Medium,High,6.834954,Medium,343.0,133.0,3.588814,19.132004,1
9996,38.0,Male,172.0,154.0,Medium,0,0,0,31.856801,1,0,1,NaN,High,8.247784,Low,377.0,83.0,2.658267,9.715709,1
9997,73.0,Male,152.0,201.0,High,1,0,1,26.899911,0,1,1,NaN,Low,4.436762,Low,248.0,88.0,4.408867,9.492429,1
9998,23.0,Male,142.0,299.0,Low,1,0,1,34.964026,1,0,1,Medium,High,8.526329,Medium,113.0,153.0,7.215634,11.873486,1


In [8]:
X = df.drop('Heart Disease Status', axis=1)
y = df['Heart Disease Status']

In [9]:
y.value_counts()

Heart Disease Status
0    8000
1    2000
Name: count, dtype: int64

In [10]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, 
                                                    stratify=y, random_state=42)
X_train.shape[0], X_test.shape[0]

(7500, 2500)

**train**

In [11]:
X_train.isna().sum()

Age                       24
Gender                    13
Blood Pressure            10
Cholesterol Level         18
Exercise Habits           21
Smoking                    0
Family Heart Disease       0
Diabetes                   0
BMI                       12
High Blood Pressure        0
Low HDL Cholesterol        0
High LDL Cholesterol       0
Alcohol Consumption     1932
Stress Level              18
Sleep Hours               15
Sugar Consumption         22
Triglyceride Level        21
Fasting Blood Sugar       17
CRP Level                 19
Homocysteine Level        17
dtype: int64

In [12]:
num_columns = X_train.select_dtypes(include=['int64', 'float64']).columns.tolist()
ordinal_columns = ['Exercise Habits', 'Stress Level', 'Sugar Consumption', 'Alcohol Consumption']
cat_columns = X_train.select_dtypes(include='object').columns.tolist()
cat_columns = [col for col in cat_columns if col not in ordinal_columns]

In [13]:
ordinal_categories = {
    'Exercise Habits': ['Low', 'Medium', 'High'],
    'Stress Level': ['Low', 'Medium', 'High'],
    'Sugar Consumption': ['Low', 'Medium', 'High'],
    'Alcohol Consumption': ['Low', 'Medium', 'High']
}
ordinal_categories_list = [ordinal_categories[col] for col in ordinal_columns]

In [14]:
num_pipeline = Pipeline(
    steps=[
        ('imputer', SimpleImputer(strategy='median')),
        ('scaler', StandardScaler())
    ]
)

ordinal_pipeline = Pipeline(
    steps=[
        ('imputer', SimpleImputer(strategy='most_frequent')),
        ('ordinal', OrdinalEncoder(categories=ordinal_categories_list))
    ]
)

cat_pipeline = Pipeline(
    steps=[
        ('imputer', SimpleImputer(strategy='most_frequent')),
        ('ohe', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
    ]
)

In [15]:
preprocessor = ColumnTransformer(
    transformers=[
        ('num', num_pipeline, num_columns),
        ('ord', ordinal_pipeline, ordinal_columns),
        ('cat', cat_pipeline, cat_columns)
    ]
)


In [16]:
X_train_processed = preprocessor.fit_transform(X_train)
X_train_processed

array([[ 1.19064359, -1.06101229,  0.23883833, ...,  1.        ,
         0.        ,  1.        ],
       [ 0.64101431,  0.9817093 , -0.68146965, ...,  1.        ,
         1.        ,  0.        ],
       [-0.07350376, -0.60707416, -0.26733106, ...,  1.        ,
         1.        ,  0.        ],
       ...,
       [-0.23839254, -0.55033189, -0.31334646, ...,  1.        ,
         0.        ,  1.        ],
       [-0.51320718,  1.60587422,  1.2971925 , ...,  2.        ,
         1.        ,  0.        ],
       [ 0.4211626 , -1.57169268,  1.2971925 , ...,  1.        ,
         1.        ,  0.        ]])

In [17]:
feature_names = preprocessor.get_feature_names_out()
X_train_df = pd.DataFrame(X_train_processed, columns=feature_names)
X_train_df.head()

,num__Age,num__Blood Pressure,num__Cholesterol Level,num__BMI,num__Sleep Hours,num__Triglyceride Level,num__Fasting Blood Sugar,num__CRP Level,num__Homocysteine Level,ord__Exercise Habits,ord__Stress Level,ord__Sugar Consumption,ord__Alcohol Consumption,cat__Gender_Female,cat__Gender_Male
0,1.190644,-1.061012,0.238838,0.909518,0.933276,1.065804,-1.328102,1.504840,-1.183552,1.0,0.0,1.0,1.0,0.0,1.0
1,0.641014,0.981709,-0.681470,-0.987970,1.535017,0.228821,0.242441,-1.628197,1.005777,2.0,2.0,0.0,1.0,1.0,0.0
2,-0.073504,-0.607074,-0.267331,-0.707937,1.215119,-0.264196,-0.097136,1.146557,1.015160,1.0,1.0,2.0,1.0,1.0,0.0
3,1.355532,0.244060,-1.440724,-0.443414,0.104812,0.022442,-0.606501,-0.363273,0.854725,1.0,2.0,2.0,1.0,1.0,0.0
4,-1.062836,0.471029,-0.911547,-0.171720,-1.013331,-1.169972,-1.710125,-0.973380,-0.067613,1.0,2.0,1.0,0.0,1.0,0.0


**test**

In [18]:
X_test_processed = preprocessor.transform(X_test)
X_test_df = pd.DataFrame(X_test_processed, columns=feature_names)
X_test_df.head()

,num__Age,num__Blood Pressure,num__Cholesterol Level,num__BMI,num__Sleep Hours,num__Triglyceride Level,num__Fasting Blood Sugar,num__CRP Level,num__Homocysteine Level,ord__Exercise Habits,ord__Stress Level,ord__Sugar Consumption,ord__Alcohol Consumption,cat__Gender_Female,cat__Gender_Male
0,1.685310,1.549132,-1.486739,0.550080,1.385789,-1.422213,-0.224477,-0.575147,-0.025542,0.0,2.0,2.0,2.0,1.0,0.0
1,-0.952911,-0.266621,-0.796508,0.546322,-1.004432,-1.089713,0.199994,1.250284,-1.385043,1.0,0.0,2.0,2.0,1.0,0.0
2,-0.183430,0.697998,-0.543423,0.929015,-0.791773,-0.206868,1.133830,-0.336801,-1.288977,1.0,0.0,2.0,1.0,1.0,0.0
3,1.685310,-0.436847,-0.819516,-0.739961,-0.414985,-0.997989,0.751806,-1.140987,0.175507,0.0,1.0,0.0,1.0,0.0,1.0
4,1.300569,-1.628435,0.468915,-1.529355,-0.185322,0.985545,0.454676,-0.454645,-0.815468,0.0,0.0,0.0,2.0,0.0,1.0


In [19]:
pd.concat([X_train_df.reset_index(drop=True), 
           y_train.reset_index(drop=True)], axis=1).to_csv(
    'data/data_train.csv',
    index=False)

pd.concat([X_test_df.reset_index(drop=True),
            y_test.reset_index(drop=True)], axis=1).to_csv(
    'data/data_test.csv',
    index=False)